In [46]:
%useLatestDescriptors
%use dataframe
%use kandy

In [47]:
val corriente = DataFrame.readCsv("Corriente.csv", delimiter = ';')
val tension = DataFrame.readCsv("Tension.csv", delimiter = ';')

In [48]:
val reference_i = corriente.get { ic }.last()
val nCorriente = corriente.update { ic }.with { it / reference_i }
nCorriente.writeCsv("corriente_escalada.csv")

val reference_v = tension.get { vce }.first() ?: 1.0
val nTension = tension.update { vce }.with { if(it!=null) it / reference_v else 1.0 }
nTension.writeCsv("tension_escalada.csv")



In [49]:
fun interp1Linear(
    x: DoubleArray,
    y: DoubleArray,
    xi: DoubleArray,
    extrapolate: Boolean = false
): DoubleArray {
    require(x.size == y.size) { "x and y must have the same length" }
    require(x.size >= 2) { "At least two data points are required" }

    // Ensure x is increasing
    val (xs, ys) = if (x[0] <= x.last()) {
        x to y
    } else {
        x.reversedArray() to y.reversedArray()
    }

    // Helper: linear interpolation between two points
    fun lerp(x1: Double, y1: Double, x2: Double, y2: Double, xq: Double): Double {
        return y1 + (xq - x1) * (y2 - y1) / (x2 - x1)
    }

    val result = DoubleArray(xi.size)

    for (i in xi.indices) {
        val xq = xi[i]

        val idx = xs.binarySearch(xq)
        if (idx >= 0) {
            // Exact match
            result[i] = ys[idx]
            continue
        }

        val ins = -idx - 1
        when {
            ins == 0 -> { // Before first point
                result[i] = if (extrapolate) lerp(xs[0], ys[0], xs[1], ys[1], xq) else 0.0
            }
            ins >= xs.size -> { // After last point
                result[i] = if (extrapolate) lerp(xs[xs.size - 2], ys[ys.size - 2], xs.last(), ys.last(), xq) else 0.0
            }
            else -> { // Inside range
                result[i] = lerp(xs[ins - 1], ys[ins - 1], xs[ins], ys[ins], xq)
            }
        }
    }

    return result
}

In [50]:
val pointCount = 100
val deltaT = 2000e-9 / pointCount
val interpT = DoubleArray(pointCount) { i -> i * deltaT}
val interpVce = interp1Linear(
    x = nTension.getColumn { t }.toTypedArray().filterNotNull().toDoubleArray(),
    y =  nTension.getColumn { vce }.toTypedArray().filterNotNull().toDoubleArray(),
    xi = interpT
)
val interpIc = interp1Linear(
    x = nCorriente.getColumn { t }.toTypedArray().filterNotNull().toDoubleArray(),
    y =  nCorriente.getColumn { ic }.toTypedArray().filterNotNull().toDoubleArray(),
    xi = interpT
)
val pInst = interpVce.zip(interpIc) { vce, ic -> vce * ic }
val E = deltaT / 2 * (pInst.first() + 2 * pInst.drop(1).dropLast(1).fold(0.0) { acc, v -> acc + v} + pInst.last())
println("Energia total de la grafica: ${String.format("%.16f", E)}")

Energia total de la grafica: 0,0000001448706351


In [51]:
run {
    val data = mapOf(
        "tiempo" to interpT.toList() + interpT.toList() + interpT.toList(),
        "y" to interpIc.toList() + interpVce.toList() + pInst.toList(),
        "legends" to List(interpT.size) { "Ic" } + List(interpT.size) { "Vce" } + List(interpT.size) { "P" }
    )  // Combine data into a map
    plot(data) { // Begin plotting
        groupBy("legends") {
            line {
                x("tiempo") { axis.name = "Corriente de Pico [A]"}
                y("y") { axis.name = "Potencia [W]" }
                color("legends")
            }
            layout { // Set plot layout
                title = "Disipacion de potencia" // Add title
                size = 1300 to 500 // Plot dimension settings
            }
        }
    }
}

<head>
 <meta charset="UTF-8">
 <style> html, body { margin: 0; overflow: hidden; } </style>
 <script type="text/javascript" data-lets-plot-script="library" src="https://cdn.jsdelivr.net/gh/JetBrains/lets-plot@v4.5.1/js-package/distr/lets-plot.min.js"></script>
 </head>
 <body>
 <div id="zNS5iL"></div>
 <script type="text/javascript" data-lets-plot-script="plot">
 
 (function() {
 // ----------
 
 var plotSpec={
"ggtitle":{
"text":"Disipacion de potencia"
},
"mapping":{
},
"data":{
},
"ggsize":{
"width":1300.0,
"height":500.0
},
"kind":"plot",
"scales":[{
"aesthetic":"x",
"name":"Corriente de Pico [A]",
"limits":[null,null]
},{
"aesthetic":"y",
"name":"Potencia [W]",
"limits":[null,null]
},{
"aesthetic":"color",
"discrete":true
}],
"layers":[{
"mapping":{
"x":"tiempo",
"y":"y",
"color":"legends",
"group":"&merged_groups"
},
"stat":"identity",
"data":{
"&merged_groups":["Ic","Ic","Ic","Ic","Ic","Ic","Ic","Ic","Ic","Ic","Ic","Ic","Ic","Ic","Ic","Ic","Ic","Ic","Ic","Ic","Ic","Ic","Ic","Ic","Ic","Ic","Ic","Ic","Ic","Ic","Ic","Ic","Ic","Ic","Ic","Ic","Ic","Ic","Ic","Ic","Ic","Ic","Ic","Ic","Ic","Ic","Ic","Ic","Ic","Ic","Ic","Ic","Ic","Ic","Ic","Ic","Ic","Ic","Ic","Ic","Ic","Ic","Ic","Ic","Ic","Ic","Ic","Ic","Ic","Ic","Ic","Ic","Ic","Ic","Ic","Ic","Ic","Ic","Ic","Ic","Ic","Ic","Ic","Ic","Ic","Ic","Ic","Ic","Ic","Ic","Ic","Ic","Ic","Ic","Ic","Ic","Ic","Ic","Ic","Ic","Vce","Vce","Vce","Vce","Vce","Vce","Vce","Vce","Vce","Vce","Vce","Vce","Vce","Vce","Vce","Vce","Vce","Vce","Vce","Vce","Vce","Vce","Vce","Vce","Vce","Vce","Vce","Vce","Vce","Vce","Vce","Vce","Vce","Vce","Vce","Vce","Vce","Vce","Vce","Vce","Vce","Vce","Vce","Vce","Vce","Vce","Vce","Vce","Vce","Vce","Vce","Vce","Vce","Vce","Vce","Vce","Vce","Vce","Vce","Vce","Vce","Vce","Vce","Vce","Vce","Vce","Vce","Vce","Vce","Vce","Vce","Vce","Vce","Vce","Vce","Vce","Vce","Vce","Vce","Vce","Vce","Vce","Vce","Vce","Vce","Vce","Vce","Vce","Vce","Vce","Vce","Vce","Vce","Vce","Vce","Vce","Vce","Vce","Vce","Vce","P","P","P","P","P","P","P","P","P","P","P","P","P","P","P","P","P","P","P","P","P","P","P","P","P","P","P","P","P","P","P","P","P","P","P","P","P","P","P","P","P","P","P","P","P","P","P","P","P","P","P","P","P","P","P","P","P","P","P","P","P","P","P","P","P","P","P","P","P","P","P","P","P","P","P","P","P","P","P","P","P","P","P","P","P","P","P","P","P","P","P","P","P","P","P","P","P","P","P","P"],
"legends":["Ic","Ic","Ic","Ic","Ic","Ic","Ic","Ic","Ic","Ic","Ic","Ic","Ic","Ic","Ic","Ic","Ic","Ic","Ic","Ic","Ic","Ic","Ic","Ic","Ic","Ic","Ic","Ic","Ic","Ic","Ic","Ic","Ic","Ic","Ic","Ic","Ic","Ic","Ic","Ic","Ic","Ic","Ic","Ic","Ic","Ic","Ic","Ic","Ic","Ic","Ic","Ic","Ic","Ic","Ic","Ic","Ic","Ic","Ic","Ic","Ic","Ic","Ic","Ic","Ic","Ic","Ic","Ic","Ic","Ic","Ic","Ic","Ic","Ic","Ic","Ic","Ic","Ic","Ic","Ic","Ic","Ic","Ic","Ic","Ic","Ic","Ic","Ic","Ic","Ic","Ic","Ic","Ic","Ic","Ic","Ic","Ic","Ic","Ic","Ic","Vce","Vce","Vce","Vce","Vce","Vce","Vce","Vce","Vce","Vce","Vce","Vce","Vce","Vce","Vce","Vce","Vce","Vce","Vce","Vce","Vce","Vce","Vce","Vce","Vce","Vce","Vce","Vce","Vce","Vce","Vce","Vce","Vce","Vce","Vce","Vce","Vce","Vce","Vce","Vce","Vce","Vce","Vce","Vce","Vce","Vce","Vce","Vce","Vce","Vce","Vce","Vce","Vce","Vce","Vce","Vce","Vce","Vce","Vce","Vce","Vce","Vce","Vce","Vce","Vce","Vce","Vce","Vce","Vce","Vce","Vce","Vce","Vce","Vce","Vce","Vce","Vce","Vce","Vce","Vce","Vce","Vce","Vce","Vce","Vce","Vce","Vce","Vce","Vce","Vce","Vce","Vce","Vce","Vce","Vce","Vce","Vce","Vce","Vce","Vce","P","P","P","P","P","P","P","P","P","P","P","P","P","P","P","P","P","P","P","P","P","P","P","P","P","P","P","P","P","P","P","P","P","P","P","P","P","P","P","P","P","P","P","P","P","P","P","P","P","P","P","P","P","P","P","P","P","P","P","P","P","P","P","P","P","P","P","P","P","P","P","P","P","P","P","P","P","P","P","P","P","P","P","P","P","P","P","P","P","P","P","P","P","P","P","P","P","P","P","P"],
"tiempo":[0.0,2.0E-8,4.0E-8,6.000000000000001E-8,8.0E-8,1.0E-7,1.2000000000000002E-7,1.4E-7,1.6E-7,1.8E

In [52]:
val df = dataFrameOf(
    "t"     to interpT.drop(1).map { it.toString() }.toList(),
    "vce"   to interpVce.drop(1).map { it.toString() }.toList(),
    "ic"    to interpIc.drop(1).map { it.toString() }.toList(),
    "p"     to pInst.drop(1).map { it.toString() }.toList()
)
df.writeCsv("encendido_igbt.csv")
df

t,vce,ic,p
2.0E-8,1.0,-0.008677299743929083,-0.008677299743929083
4.0E-8,1.0,-0.009988977987298257,-0.009988977987298257
6.000000000000001E-8,1.0,-0.011106532813904705,-0.011106532813904705
8.0E-8,1.0,-0.01103161522325442,-0.01103161522325442
1.0E-7,0.996677796327212,-0.010956697632604134,-0.010920297251487469
1.2000000000000002E-7,0.993338898163606,-0.010883327520055806,-0.010810832567125884
1.4E-7,0.99,-0.010813568189745376,-0.010705432507847923
1.6E-7,0.99,-0.010743808859434949,-0.010636370770840599
1.8E-7,0.99,-0.010672185998627315,-0.010565464138641041
2.0E-7,0.99,-0.01059497254632807,-0.01048902282086479


In [53]:
val corriente = DataFrame.readCsv("Corriente_Apagado.csv", delimiter = ';')
val tension = DataFrame.readCsv("Tension_Apagado.csv", delimiter = ';')

In [54]:
val reference_i = corriente.get { ic }.first()
val nCorriente = corriente.update { ic }.with { it / reference_i }
nCorriente.writeCsv("corriente_escalada_apagado.csv")

val reference_v = tension.get { vce }.last()
val nTension = tension.update { vce }.with { it / reference_v }
nTension.writeCsv("tension_escalada_apagado.csv")

In [55]:
val pointCount = 100
val deltaT = 2000e-9 / pointCount
val interpT = DoubleArray(pointCount) { i -> i * deltaT}
val interpVce = interp1Linear(
    x = nTension.getColumn { t }.toTypedArray().filterNotNull().toDoubleArray(),
    y =  nTension.getColumn { vce }.toTypedArray().filterNotNull().toDoubleArray(),
    xi = interpT
)
val interpIc = interp1Linear(
    x = nCorriente.getColumn { t }.toTypedArray().filterNotNull().toDoubleArray(),
    y =  nCorriente.getColumn { ic }.toTypedArray().filterNotNull().toDoubleArray(),
    xi = interpT
)
val pInst = interpVce.zip(interpIc) { vce, ic -> vce * ic }
val E = deltaT / 2 * (pInst.first() + 2 * pInst.drop(1).dropLast(1).fold(0.0) { acc, v -> acc + v} + pInst.last())
println("Energia total de la grafica: ${String.format("%.16f", E)}")

Energia total de la grafica: 0,0000002921576001


In [56]:
run {
    val data = mapOf(
        "tiempo" to interpT.toList() + interpT.toList() + interpT.toList(),
        "y" to interpIc.toList() + interpVce.toList() + pInst.toList(),
        "legends" to List(interpT.size) { "Ic" } + List(interpT.size) { "Vce" } + List(interpT.size) { "P" }
    )  // Combine data into a map
    plot(data) { // Begin plotting
        groupBy("legends") {
            line {
                x("tiempo") { axis.name = "Corriente de Pico [A]"}
                y("y") { axis.name = "Potencia [W]" }
                color("legends")
            }
            layout { // Set plot layout
                title = "Disipacion de potencia" // Add title
                size = 1300 to 500 // Plot dimension settings
            }
        }
    }
}

<head>
 <meta charset="UTF-8">
 <style> html, body { margin: 0; overflow: hidden; } </style>
 <script type="text/javascript" data-lets-plot-script="library" src="https://cdn.jsdelivr.net/gh/JetBrains/lets-plot@v4.5.1/js-package/distr/lets-plot.min.js"></script>
 </head>
 <body>
 <div id="kVqlaR"></div>
 <script type="text/javascript" data-lets-plot-script="plot">
 
 (function() {
 // ----------
 
 var plotSpec={
"ggtitle":{
"text":"Disipacion de potencia"
},
"mapping":{
},
"data":{
},
"ggsize":{
"width":1300.0,
"height":500.0
},
"kind":"plot",
"scales":[{
"aesthetic":"x",
"name":"Corriente de Pico [A]",
"limits":[null,null]
},{
"aesthetic":"y",
"name":"Potencia [W]",
"limits":[null,null]
},{
"aesthetic":"color",
"discrete":true
}],
"layers":[{
"mapping":{
"x":"tiempo",
"y":"y",
"color":"legends",
"group":"&merged_groups"
},
"stat":"identity",
"data":{
"&merged_groups":["Ic","Ic","Ic","Ic","Ic","Ic","Ic","Ic","Ic","Ic","Ic","Ic","Ic","Ic","Ic","Ic","Ic","Ic","Ic","Ic","Ic","Ic","Ic","Ic","Ic","Ic","Ic","Ic","Ic","Ic","Ic","Ic","Ic","Ic","Ic","Ic","Ic","Ic","Ic","Ic","Ic","Ic","Ic","Ic","Ic","Ic","Ic","Ic","Ic","Ic","Ic","Ic","Ic","Ic","Ic","Ic","Ic","Ic","Ic","Ic","Ic","Ic","Ic","Ic","Ic","Ic","Ic","Ic","Ic","Ic","Ic","Ic","Ic","Ic","Ic","Ic","Ic","Ic","Ic","Ic","Ic","Ic","Ic","Ic","Ic","Ic","Ic","Ic","Ic","Ic","Ic","Ic","Ic","Ic","Ic","Ic","Ic","Ic","Ic","Ic","Vce","Vce","Vce","Vce","Vce","Vce","Vce","Vce","Vce","Vce","Vce","Vce","Vce","Vce","Vce","Vce","Vce","Vce","Vce","Vce","Vce","Vce","Vce","Vce","Vce","Vce","Vce","Vce","Vce","Vce","Vce","Vce","Vce","Vce","Vce","Vce","Vce","Vce","Vce","Vce","Vce","Vce","Vce","Vce","Vce","Vce","Vce","Vce","Vce","Vce","Vce","Vce","Vce","Vce","Vce","Vce","Vce","Vce","Vce","Vce","Vce","Vce","Vce","Vce","Vce","Vce","Vce","Vce","Vce","Vce","Vce","Vce","Vce","Vce","Vce","Vce","Vce","Vce","Vce","Vce","Vce","Vce","Vce","Vce","Vce","Vce","Vce","Vce","Vce","Vce","Vce","Vce","Vce","Vce","Vce","Vce","Vce","Vce","Vce","Vce","P","P","P","P","P","P","P","P","P","P","P","P","P","P","P","P","P","P","P","P","P","P","P","P","P","P","P","P","P","P","P","P","P","P","P","P","P","P","P","P","P","P","P","P","P","P","P","P","P","P","P","P","P","P","P","P","P","P","P","P","P","P","P","P","P","P","P","P","P","P","P","P","P","P","P","P","P","P","P","P","P","P","P","P","P","P","P","P","P","P","P","P","P","P","P","P","P","P","P","P"],
"legends":["Ic","Ic","Ic","Ic","Ic","Ic","Ic","Ic","Ic","Ic","Ic","Ic","Ic","Ic","Ic","Ic","Ic","Ic","Ic","Ic","Ic","Ic","Ic","Ic","Ic","Ic","Ic","Ic","Ic","Ic","Ic","Ic","Ic","Ic","Ic","Ic","Ic","Ic","Ic","Ic","Ic","Ic","Ic","Ic","Ic","Ic","Ic","Ic","Ic","Ic","Ic","Ic","Ic","Ic","Ic","Ic","Ic","Ic","Ic","Ic","Ic","Ic","Ic","Ic","Ic","Ic","Ic","Ic","Ic","Ic","Ic","Ic","Ic","Ic","Ic","Ic","Ic","Ic","Ic","Ic","Ic","Ic","Ic","Ic","Ic","Ic","Ic","Ic","Ic","Ic","Ic","Ic","Ic","Ic","Ic","Ic","Ic","Ic","Ic","Ic","Vce","Vce","Vce","Vce","Vce","Vce","Vce","Vce","Vce","Vce","Vce","Vce","Vce","Vce","Vce","Vce","Vce","Vce","Vce","Vce","Vce","Vce","Vce","Vce","Vce","Vce","Vce","Vce","Vce","Vce","Vce","Vce","Vce","Vce","Vce","Vce","Vce","Vce","Vce","Vce","Vce","Vce","Vce","Vce","Vce","Vce","Vce","Vce","Vce","Vce","Vce","Vce","Vce","Vce","Vce","Vce","Vce","Vce","Vce","Vce","Vce","Vce","Vce","Vce","Vce","Vce","Vce","Vce","Vce","Vce","Vce","Vce","Vce","Vce","Vce","Vce","Vce","Vce","Vce","Vce","Vce","Vce","Vce","Vce","Vce","Vce","Vce","Vce","Vce","Vce","Vce","Vce","Vce","Vce","Vce","Vce","Vce","Vce","Vce","Vce","P","P","P","P","P","P","P","P","P","P","P","P","P","P","P","P","P","P","P","P","P","P","P","P","P","P","P","P","P","P","P","P","P","P","P","P","P","P","P","P","P","P","P","P","P","P","P","P","P","P","P","P","P","P","P","P","P","P","P","P","P","P","P","P","P","P","P","P","P","P","P","P","P","P","P","P","P","P","P","P","P","P","P","P","P","P","P","P","P","P","P","P","P","P","P","P","P","P","P","P"],
"tiempo":[0.0,2.0E-8,4.0E-8,6.000000000000001E-8,8.0E-8,1.0E-7,1.2000000000000002E-7,1.4E-7,1.6E-7,1.8E

In [58]:
val df = dataFrameOf(
    "t"     to interpT.drop(1).map { it.toString() }.toList(),
    "vce"   to interpVce.drop(1).map { it.toString() }.toList(),
    "ic"    to interpIc.drop(1).map { it.toString() }.toList(),
    "p"     to pInst.drop(1).map { it.toString() }.toList()
)
df.writeCsv("apagado_igbt.csv")
df

t,vce,ic,p
2.0E-8,0.00460832937020178,1.0,0.00460832937020178
4.0E-8,0.00460832937020178,1.0,0.00460832937020178
6.000000000000001E-8,0.00460832937020178,1.0,0.00460832937020178
8.0E-8,0.00460832937020178,1.0,0.00460832937020178
1.0E-7,0.0057248286571929385,1.0,0.0057248286571929385
1.2000000000000002E-7,0.007422797308993045,1.0,0.007422797308993045
1.4E-7,0.00912076596079315,1.0,0.00912076596079315
1.6E-7,0.010818734612593254,1.0,0.010818734612593254
1.8E-7,0.012516703264393358,1.0,0.012516703264393358
2.0E-7,0.014156645534433382,1.0,0.014156645534433382
